# 🔬 EKF & Zone Model Verification
**Purpose:** Validate the theoretical RC zone model and EKF estimator against EnergyPlus ground truth.  
For a single zone, compare: **Simulation** (solid) vs **Theoretical** (dashed) vs **EKF** (dotted).

In [1]:
import sys
sys.path.insert(0, '/home/jazz/Projects/Statistical-Learning-e20452')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from data_wrangling import DataInspector, DataPlotter

CSV_PATH = './results/state_log.csv'
df = pd.read_csv(CSV_PATH)

# Build datetime index (EnergyPlus uses Hour=24 for midnight → roll to next day)
base_year = 2014
def _ep_to_datetime(row):
    day, hour, minute = int(row['DayOfYear']), int(row['Hour']), int(row['Minute'])
    if hour >= 24:
        day += 1
        hour -= 24
    return pd.Timestamp(year=base_year, month=1, day=1) + pd.Timedelta(days=day-1, hours=hour, minutes=minute)

df['Datetime'] = df.apply(_ep_to_datetime, axis=1)

ZONES = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
print(f'Loaded {len(df)} timesteps')

Loaded 2880 timesteps


In [2]:
# ── SELECT ZONE HERE ─────────────────────────────────────────────────────
ZONE = 'SPACE1-1'
# ──────────────────────────────────────────────────────────────────────────

## 1 · Temperature Verification (T_in & T_m)
Compare Simulation → Theoretical RC model → EKF estimate.

In [3]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=['Indoor Air Temperature (T_in)', 'Mass Temperature (T_m)'],
    vertical_spacing=0.10
)

t = df['Datetime']

# T_in
fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_Temp_C'],
    name='Sim T_in', line=dict(color='#EF553B', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_T_in_theo'],
    name='Theo T_in', line=dict(color='#EF553B', width=1.5, dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_T_in_est'],
    name='EKF T_in', line=dict(color='#FECB52', width=1.5, dash='dot')), row=1, col=1)

# T_m
fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_T_m_C'],
    name='Sim T_m', line=dict(color='#FFA15A', width=2)), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_T_m_theo'],
    name='Theo T_m', line=dict(color='#FFA15A', width=1.5, dash='dash')), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_T_m_est'],
    name='EKF T_m', line=dict(color='#FECB52', width=1.5, dash='dot')), row=2, col=1)

fig.update_layout(template='plotly_dark', height=550,
    title=f'Temperature Verification — {ZONE}',
    legend=dict(orientation='h', y=-0.10))
fig.update_yaxes(title_text='°C', row=1, col=1)
fig.update_yaxes(title_text='°C', row=2, col=1)
fig.show()

/home/jazz/.local/lib/python3.14/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



## 2 · Humidity Verification (W_in)

In [4]:
fig = go.Figure()
t = df['Datetime']

fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_W_in_kg_kg'],
    name='Sim W_in', line=dict(color='#19D3F3', width=2)))
fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_W_in_theo'],
    name='Theo W_in', line=dict(color='#19D3F3', width=1.5, dash='dash')))
fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_W_in_est'],
    name='EKF W_in', line=dict(color='#FECB52', width=1.5, dash='dot')))

fig.update_layout(template='plotly_dark', height=400,
    title=f'Humidity Ratio Verification — {ZONE}',
    yaxis_title='kg_water / kg_air',
    legend=dict(orientation='h', y=-0.15))
fig.show()

## 3 · CO₂ Verification (C_in)

In [5]:
fig = go.Figure()
t = df['Datetime']

fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_CO2_ppm'],
    name='Sim CO₂', line=dict(color='#B6E880', width=2)))
fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_C_in_theo'],
    name='Theo CO₂', line=dict(color='#B6E880', width=1.5, dash='dash')))
fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_C_in_est'],
    name='EKF CO₂', line=dict(color='#FECB52', width=1.5, dash='dot')))

fig.update_layout(template='plotly_dark', height=400,
    title=f'CO₂ Concentration Verification — {ZONE}',
    yaxis_title='ppm',
    legend=dict(orientation='h', y=-0.15))
fig.show()

## 4 · EKF Occupancy Estimation vs Ground Truth

In [6]:
fig = go.Figure()
t = df['Datetime']

fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_Occupants'],
    name='Actual Occupancy', line=dict(color='#FF6692', width=2, dash='dot')))
fig.add_trace(go.Scatter(x=t, y=df[f'{ZONE}_Occ_est'],
    name='EKF Estimated Occ', line=dict(color='#FECB52', width=1.5)))

fig.update_layout(template='plotly_dark', height=400,
    title=f'Occupancy Estimation — {ZONE}',
    yaxis_title='People',
    legend=dict(orientation='h', y=-0.15))
fig.show()

## 5 · Error Analysis

In [7]:
# Compute errors
err_theo_T  = df[f'{ZONE}_Temp_C'] - df[f'{ZONE}_T_in_theo']
err_ekf_T   = df[f'{ZONE}_Temp_C'] - df[f'{ZONE}_T_in_est']
err_theo_W  = df[f'{ZONE}_W_in_kg_kg'] - df[f'{ZONE}_W_in_theo']
err_ekf_W   = df[f'{ZONE}_W_in_kg_kg'] - df[f'{ZONE}_W_in_est']
err_theo_C  = df[f'{ZONE}_CO2_ppm'] - df[f'{ZONE}_C_in_theo']
err_ekf_C   = df[f'{ZONE}_CO2_ppm'] - df[f'{ZONE}_C_in_est']

print(f'=== Error Summary for {ZONE} ===')
print(f'\nTemperature (°C):')
print(f'  Theoretical  →  MAE: {err_theo_T.abs().mean():.4f},  RMSE: {np.sqrt((err_theo_T**2).mean()):.4f}')
print(f'  EKF          →  MAE: {err_ekf_T.abs().mean():.4f},  RMSE: {np.sqrt((err_ekf_T**2).mean()):.4f}')
print(f'\nHumidity (kg/kg):')
print(f'  Theoretical  →  MAE: {err_theo_W.abs().mean():.6f},  RMSE: {np.sqrt((err_theo_W**2).mean()):.6f}')
print(f'  EKF          →  MAE: {err_ekf_W.abs().mean():.6f},  RMSE: {np.sqrt((err_ekf_W**2).mean()):.6f}')
print(f'\nCO₂ (ppm):')
print(f'  Theoretical  →  MAE: {err_theo_C.abs().mean():.2f},  RMSE: {np.sqrt((err_theo_C**2).mean()):.2f}')
print(f'  EKF          →  MAE: {err_ekf_C.abs().mean():.2f},  RMSE: {np.sqrt((err_ekf_C**2).mean()):.2f}')

=== Error Summary for SPACE1-1 ===

Temperature (°C):
  Theoretical  →  MAE: 1.3799,  RMSE: 1.8158
  EKF          →  MAE: 0.1785,  RMSE: 0.7754

Humidity (kg/kg):
  Theoretical  →  MAE: 0.000196,  RMSE: 0.000392
  EKF          →  MAE: 0.000131,  RMSE: 0.000284

CO₂ (ppm):
  Theoretical  →  MAE: 4.97,  RMSE: 17.12
  EKF          →  MAE: 6.90,  RMSE: 22.35


In [8]:
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=['T_in Error (Sim − Model)', 'W_in Error', 'CO₂ Error'],
    vertical_spacing=0.08
)
t = df['Datetime']

fig.add_trace(go.Scatter(x=t, y=err_theo_T, name='Theo err', line=dict(color='#EF553B', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=err_ekf_T, name='EKF err', line=dict(color='#FECB52', width=1)), row=1, col=1)

fig.add_trace(go.Scatter(x=t, y=err_theo_W, name='Theo err', line=dict(color='#19D3F3', width=1), showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=err_ekf_W, name='EKF err', line=dict(color='#FECB52', width=1), showlegend=False), row=2, col=1)

fig.add_trace(go.Scatter(x=t, y=err_theo_C, name='Theo err', line=dict(color='#B6E880', width=1), showlegend=False), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=err_ekf_C, name='EKF err', line=dict(color='#FECB52', width=1), showlegend=False), row=3, col=1)

fig.update_layout(template='plotly_dark', height=650,
    title=f'Estimation Error Over Time — {ZONE}',
    legend=dict(orientation='h', y=-0.06))
fig.update_yaxes(title_text='°C', row=1, col=1)
fig.update_yaxes(title_text='kg/kg', row=2, col=1)
fig.update_yaxes(title_text='ppm', row=3, col=1)
fig.show()

## 6 · Error Distribution (data_wrangling)

In [9]:
err_df = pd.DataFrame({
    'Theo_T_err': err_theo_T,
    'EKF_T_err': err_ekf_T,
    'Theo_CO2_err': err_theo_C,
    'EKF_CO2_err': err_ekf_C
})

di = DataInspector()
di.df = err_df
di.summary_plot(
    columns=['Theo_T_err', 'EKF_T_err', 'Theo_CO2_err', 'EKF_CO2_err'],
    numeric_plots=['violin', 'histogram'],
    separate_plots=True
)